# Compsognathus: learn quiet bilateral balance

[Open in Colab](https://colab.research.google.com/github/kuds/mesozoic-labs/blob/codex/compsognathus-foot-research/notebooks/compsognathus_balance_study.ipynb)

Choose **Runtime → Run all**. The default full workflow reads your existing Drive experiments, replays the compatible saved Compsognathus model over 40 episodes, calibrates 40 home-controller episodes, checks all four training arms with short probes, and then runs all **12 full arm/seed combinations**. It saves progress to Drive and resumes persisted training when Run all is used again with the same study name.

Source loads directly from PR527; no source ZIP or model upload is needed. Existing experiment files are read in place, and new outputs go to a separate study folder. A = current reward/filter off; B = current reward/10 Hz; C = bilateral reward/filter off; D = bilateral reward/10 Hz. Each full arm uses seeds 42, 43 and 44. The new arms start fresh; historical checkpoints remain comparison baselines with their original identities.

The complete training suite contains **132,055,040 environment steps**, including four 8,192-step probes and PPO's full-update rounding. The previous Compsognathus 11M run took 13h 32m 53s, suggesting **162h 34m 36s — about 163 hours of runtime (6.8 days)** for 12 full runs before additional checks. The historical run used a Colab L4 runtime; this study uses CPU training and adds physics-rate evaluations, so this is a planning estimate, not a runtime guarantee. One uninterrupted Colab session is unlikely to cover the suite. Keep the same study name and rerun Run all in a matching runtime to continue from saved progress. Normal interruption runs the Drive flush; abrupt runtime loss may prevent it, so the workflow also saves checkpoints throughout training.


In [ ]:
# @title Choose the workflow
RUN_MODE = "full"  # @param ["full", "smoke"]
STUDY_NAME = "compsognathus-balance-v1"  # @param {type:"string"}
COMPY_REFERENCE_RUN = "20260909_162812"  # @param {type:"string"}
TREX_REFERENCE_RUN = "20260821_142144"  # @param {type:"string"}

# full: historical references, 40-episode home calibration, four probes, then all 12 full runs.
# smoke: the same references and calibration, followed by the four probes only.
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("RUN_MODE must be full or smoke")

In [ ]:
# @title Mount existing Drive and load source directly from the PR
import json
import os
import re
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive did not mount; persistent storage is required")
BASE = Path("/content/drive/MyDrive/mesozoic-labs")
if not STUDY_NAME or STUDY_NAME in {".", ".."} or Path(STUDY_NAME).name != STUDY_NAME:
    raise ValueError("STUDY_NAME must be one folder name")
STUDY = BASE / STUDY_NAME
STUDY.mkdir(parents=True, exist_ok=True)
token = str(uuid.uuid4())
storage_check = STUDY / (".write-check-" + token)
try:
    with storage_check.open("w") as stream:
        stream.write(token)
        stream.flush()
        os.fsync(stream.fileno())
    if storage_check.read_text() != token:
        raise IOError("Drive write verification failed")
finally:
    storage_check.unlink(missing_ok=True)

EXPECTED_IMPLEMENTATION = "sha256:d282c3d805592075efb9a0e3e237b6e2461983442270856c5a9062c3061435b4"
saved_plan = STUDY / "study_plan.json"
saved = json.loads(saved_plan.read_text()) if saved_plan.exists() else None
revision = saved["source_commit"] if saved else "refs/pull/527/head"
if saved and (not re.fullmatch(r"[0-9a-f]{40}", revision) or saved["implementation_sha256"] != EXPECTED_IMPLEMENTATION):
    raise RuntimeError("Open the notebook snapshot saved with this study, or use a new study name for changed source")

REPO = Path("/content/mesozoic-balance-pr527")
REMOTE = "https://github.com/kuds/mesozoic-labs.git"


def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()


if not (REPO / ".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError("Source directory contains other files; use a fresh runtime")
    REPO.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "init", "-q", str(REPO)])
    git("remote", "add", "origin", REMOTE)
if git("remote", "get-url", "origin") != REMOTE:
    raise RuntimeError("Source checkout has an unexpected remote")
if git("status", "--porcelain"):
    raise RuntimeError("Source checkout contains changes; use a fresh runtime")
git("fetch", "--quiet", "--depth=1", "origin", revision)
target = git("rev-parse", "FETCH_HEAD")
if any(name == "environments" or name.startswith("environments.") for name in sys.modules):
    if git("rev-parse", "HEAD") != target:
        raise RuntimeError("Study modules are already imported from another revision; use a fresh runtime")
git("checkout", "--quiet", "--detach", target)
os.chdir(REPO)
print("Source revision:", target)
print("Existing experiments:", BASE / "logs")
print("New persistent results:", STUDY)

In [ ]:
# @title Install the training dependencies
import subprocess
import sys

# On a later session, restore the recorded core package versions before import.
# Python itself is supplied by Colab; a changed Python needs a matching runtime.
install = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[train]"]
saved_plan = STUDY / "study_plan.json"
if saved_plan.exists():
    runtime = json.loads(saved_plan.read_text())["runtime"]
    if sys.version.split()[0] != runtime["python"]:
        raise RuntimeError(
            "This study requires Python "
            + runtime["python"]
            + "; this runtime has "
            + sys.version.split()[0]
            + ". Use a Colab runtime with the recorded Python version. Do not mix results across runtimes."
        )
    constraints = Path("/content/balance-study-constraints.txt")
    constraints.write_text("\n".join(name + "==" + value for name, value in runtime.items() if name != "python") + "\n")
    install.extend(["--constraint", str(constraints)])
subprocess.check_call(install)
sys.path.insert(0, str(REPO))
from environments.compsognathus.experiments.balance_identity import study_source_fingerprint

if study_source_fingerprint() != EXPECTED_IMPLEMENTATION:
    raise ValueError("Installed study source does not match this notebook")

In [ ]:
# @title Prepare the saved-reference and calibration steps
# These run inside the workflow's try/finally so Drive is flushed on completion or interruption.
def prepare_references_and_calibration():
    from environments.compsognathus.experiments.balance_drive import (
        discover_balance_runs,
        evaluate_saved_baseline,
        save_historical_baselines,
    )
    from environments.compsognathus.scripts.train_balance_study import prepare_study
    from environments.shared.plant_contract import PlantContractError

    plan = prepare_study(STUDY)
    print("Arms:", list(plan["arms"]))
    print("Training seeds:", plan["training_seeds"])
    print("Proposed physical targets:", plan["behavior_targets"])
    print("Package versions:", plan["runtime"])
    snapshot = STUDY / "compsognathus_balance_study.ipynb"
    if not snapshot.exists():
        snapshot.write_bytes((REPO / "notebooks/compsognathus_balance_study.ipynb").read_bytes())

    historical_runs = discover_balance_runs(BASE)
    save_historical_baselines(historical_runs, STUDY / "references" / "historical_baselines.json")
    print("Discovered saved stance runs:", len(historical_runs))
    selected_references = {}
    for species, run_id in {"compsognathus": COMPY_REFERENCE_RUN, "trex": TREX_REFERENCE_RUN}.items():
        matches = [row for row in historical_runs if row["species"] == species and row["run_id"] == run_id]
        if not matches:
            print(
                "Reference not found:",
                species,
                run_id,
                "— available:",
                [row["run_id"] for row in historical_runs if row["species"] == species],
            )
            continue
        selected_references[species] = matches[0]
        print(species, run_id, matches[0]["status"], matches[0].get("saved_metrics", {}))
        print("Saved selected pair:", matches[0]["checkpoint_pair"])

    reference = selected_references.get("compsognathus")
    pair = reference["checkpoint_pair"] if reference else None
    if pair:
        replay_root = STUDY / "references" / "replays" / (COMPY_REFERENCE_RUN + "_n40")
        replay_seeds = tuple(range(11042, 11082))
        # Reuse a completed replay; preserve incomplete attempts and retry them in a new folder.
        completed = [replay_root / "baseline.json"] + sorted(replay_root.glob("attempt_*/baseline.json"))
        baseline = None
        for path in completed:
            if not path.exists():
                continue
            candidate = json.loads(path.read_text())
            if candidate["pair"] != pair:
                raise ValueError("Saved Drive model/config changed since replay; use a new study folder")
            if tuple(row["seed"] for row in candidate["episodes"]) != replay_seeds:
                raise ValueError("Saved baseline replay uses another episode panel")
            baseline = candidate
            print("Reusing completed 40-episode baseline replay:", baseline["summary"])
            break
        if baseline is None:
            replay_root.mkdir(parents=True, exist_ok=True)
            attempt = 1
            while (replay_root / ("attempt_" + str(attempt))).exists():
                attempt += 1
            try:
                baseline = evaluate_saved_baseline(pair, replay_root / ("attempt_" + str(attempt)), seeds=replay_seeds)
                print("Existing Compy policy with new physical measurements:", baseline["summary"])
            except PlantContractError as exc:
                print("Saved model replay blocked by compatibility verification:", str(exc))
                print("The existing reports remain available. No checkpoint identity was changed.")
    else:
        print("Saved reports imported; replay requires an explicitly recorded, available Compy pair.")
    print("Historical control-window support and new physics-level load metrics use different definitions.")

    if not (STUDY / "calibration.json").exists():
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "environments.compsognathus.scripts.train_balance_study",
                "calibrate",
                "--output",
                str(STUDY),
            ]
        )
    calibration = json.loads((STUDY / "calibration.json").read_text())
    home = calibration["home_reference"]
    print("Home reference complete episodes:", sum(row["full_horizon"] for row in home), "/", len(home))

## Run and continue the whole study

The next cell automatically checks all four probes before starting the 12 full runs. A failed probe prevents full training. Full jobs use the same 11M-step budget, four environments, and screening every 250k steps. Screening chooses a checkpoint; its exact saved model and normalization files are then loaded for the separate 40-seed confirmation panel. A short probe cannot qualify learned balance.

Completed jobs are validated and skipped, interrupted jobs resume from saved progress, and failures stay visible in the suite reports. Checkpoints are saved approximately every 250k steps. Resume preserves policy weights, optimizer state, normalization statistics and the original full schedules, while resetting simulator episodes. It is not a bit-for-bit continuation of the physical trajectory; unsaved work may be repeated. Run all again with the same study name after a runtime restart. The prepared plan checks source and package versions before continuing. Training uses CPU for this small policy, even if Colab provides a GPU.


In [ ]:
# @title Run all references, calibration and training; always flush Drive afterward
from environments.compsognathus.experiments.balance_suite import run_balance_suite

try:
    prepare_references_and_calibration()
    suite = run_balance_suite(STUDY, mode=RUN_MODE)
    print("Suite status:", suite.get("status"))
    print("All requested jobs complete:", suite.get("all_jobs_complete"))
    comparison = suite.get("comparison", {})
    for row in comparison.get("runs", []):
        print(row["arm"], row["seed"], row["status"], "balance qualified:", row["learned_balance_qualified"])
    print("Persistent progress:", STUDY / "suite_progress.json")
    print("Suite results:", STUDY / "suite_comparison.json")
    print("Full-run comparison:", STUDY / "comparison.json")
finally:
    print("Flushing pending Drive writes before this session ends...")
    drive.flush_and_unmount(timeout_ms=300000)
    print("Drive flushed and unmounted. Run all again to remount and continue saved work.")

## Read the results

`suite_progress.json` tracks every planned probe and full run. `suite_comparison.json` and `comparison.json` include completed, failed and unfinished jobs, so a missing run cannot disappear from the comparison. Existing-model references live under `references/`; `calibration.json` records the home-controller comparison.

Within each full run, `run_summary.json` records whether the selected checkpoint met the proposed behavior targets. `confirmation.json` contains all confirmation episodes, the unchanged stance criteria projected onto canonical reward, and the additional balance criteria. `screening_history.json` records checkpoint selection. `updates.json` records learning rate, exploration coefficient, learned action standard deviation, and optimizer diagnostics. `selected_trace.csv` contains 50 Hz control-boundary traces; physical balance aggregates sample every 2 ms physics step.

Compare physical behavior across reward variants, rather than raw training return. The full-run confirmation panel is separate from training, screening and the workflow-probe panels. A pass here is a research result. Disturbance recovery, deliberate foot unloading, and production promotion remain separate validation steps.
